# Schachprojekt – Codeübersicht

## Inhaltsverzeichnis
- [Übersicht](#übersicht)
- [Klassen und ihre Aufgaben](#klassen-und-ihre-aufgaben)
  - [Color](#color)
  - [Player](#player)
  - [Board](#board)
  - [Move](#move)
  - [Piece und abgeleitete Klassen](#piece-und-abgeleitete-klassen)
- [Spielablauf und Beispiele](#spielablauf-und-beispiele)
- [Hinweise und mögliche Erweiterungen](#hinweise-und-mögliche-erweiterungen)

---

## Übersicht

Dieses Projekt bildet die Grundlage für ein Schachspiel in Python. Mittels objektorientierter Programmierung werden zentrale Bausteine wie das Spielbrett, die Figuren, Spieler und Züge modelliert. Jede Figur besitzt eine Referenz auf das zugehörige Board, sodass Methoden der Figuren direkt den Zustand des Spielbretts abfragen können. Ein zentrales Element ist die Simulation von Zügen über eine Deepcopy des Boards, um zu prüfen, ob ein Zug den eigenen König gefährdet.

---

## Klassen und ihre Aufgaben

### Color
- **Beschreibung:**  
  Eine Enum-Klasse, die die beiden Spielerfarben definiert: `WHITE` und `BLACK`.
- **Verwendung:**  
  Dient zur Farbzuteilung von Spielern und Figuren.

### Player
- **Beschreibung:**  
  Repräsentiert einen Spieler.
- **Attribute:**  
  - `name`: Name des Spielers.  
  - `color`: Zugewiesene Farbe (vom Typ `Color`).

### Board
- **Beschreibung:**  
  Repräsentiert das Schachbrett und verwaltet den Zustand des Spiels.
- **Wichtige Attribute und Methoden:**  
  - `grid`: Ein 8×8-Array, das die Positionen der Figuren speichert.  
  - `pieces_on_board`: Liste aller aktiven Figuren.  
  - `setup_pieces(color)`: Platziert Figuren der übergebenen Farbe an den festgelegten Startpositionen.  
  - `reset_grid()` und `update_grid()`: Aktualisieren die Darstellung des Brettes, indem das Raster neu belegt wird.  
  - `initialize()`: Führt die initiale Einrichtung des Brettes durch (Befüllen der Figuren für beide Farben und Aktualisieren des Grids).  
  - `is_empty(square_coords)`: Prüft, ob ein bestimmtes Feld leer ist.  
  - `in_bounds(square_coords)`: Stellt sicher, dass angegebene Koordinaten innerhalb der Brettgrenzen liegen.  
  - `clone()`: Erstellt über `copy.deepcopy` eine Kopie des Boards, um Zugsimulationen zu ermöglichen.  
  - `is_under_attack(color)`: (Platzhalter) Prüft, ob der König der angegebenen Farbe angegriffen wird.

### Move
- **Beschreibung:**  
  Repräsentiert einen Zug im Spiel.
- **Attribute:**  
  - `captured_pieces`: Liste von Figuren, die durch den Zug geschlagen wurden.  
  - `move_history`: Historie der bisher ausgeführten Züge.

### Piece und abgeleitete Klassen
- **Basis-Klasse (Piece):**
  - **Beschreibung:**  
    Abstrakte Klasse, von der alle Schachfiguren erben.
  - **Attribute:**  
    - `_color`: Farbe der Figur.  
    - `_position`: Aktuelle Position als Tupel (x, y).  
    - `_has_moved`: Gibt an, ob die Figur bereits bewegt wurde (wichtig für spezielle Züge wie Rochade oder den doppelten Bauernzug).  
    - `board`: Referenz auf das zugehörige Board.
  - **Wichtige Methoden:**  
    - `get_pseudo_legal_moves()`: (Abstract) Gibt alle möglichen Züge gemäß den Bewegungsregeln der Figur zurück, ohne Berücksichtigung von Schachbedingungen.  
    - `get_legal_moves()`: Filtert die pseudo-legalen Züge, indem für jeden Zug eine Simulation (über `clone()`) durchgeführt wird, um zu prüfen, ob der eigene König damit in Schach gerät.  
    - `move(end_position)`: Führt einen Zug aus, wenn dieser als legal erkannt wird.
- **Spezialisierte Klassen (z. B. Pawn, Rook, Knight, Bishop, Queen, King):**
  - Jede dieser Klassen implementiert in der Methode `get_pseudo_legal_moves()` die spezifische Zuglogik.
  - **Beispiel – Pawn:**
    - Berechnet Standardzüge (ein Feld vorwärts), den Doppelzug vom Startfeld, diagonale Schläge und den en passant-Zug.
    - Verwendet das Flag `_en_passant_vulnerability` zur Kennzeichnung spezieller en passant-Situationen.

---

## Spielablauf und Beispiele

Das Spiel wird über die **Board**-Klasse initialisiert. Dabei werden alle Figuren an ihre Startpositionen gesetzt und das interne Grid aktualisiert. Züge werden direkt über die Figuren-Methoden ausgeführt.

### Beispiel: Erster Bauernzug

```python
# Initialisiere das Spielbrett und setze die Figuren
board = Board()
board.initialize()

# Bewege den Bauern an Position (0, 1) (das Feld links vom weißen Bauern) nach (0, 2)
board.grid[1][0].move((0, 2))

# Aktualisiere das Brett, um die neue Position anzuzeigen
board.update_grid()


In [56]:
from enum import Enum
from abc import ABC, abstractmethod
import copy


To do:

ich mache es ja so dass ich vin jeder figur die position als eigenschaft des klassenobjekts speichere. jetzt muss ich noch einbauen, dass die frühere figur verschwindet wenn sie geschlagen wurde.

v2

In [57]:
class Color(Enum):
    WHITE = "white"
    BLACK = "black"



class Player():
    def __init__(self, name, color):
        self.name = name
        self.color = color



class Board():
    def __init__(self):
        self.grid = [
        [None for _ in range(8)] for _ in range(8)
        ]
        self.white_king_position = (0, 4)
        self.black_king_postition = (7, 4)
        self.pieces_on_board = []
    
    
    def setup_pieces(self, color):
        
        row_pawns = 1 if color == Color.WHITE else 6
        row_others = 0 if color == Color.WHITE else 7

        for col in range (8):
            self.pieces_on_board.append(Pawn(color, position=(col, row_pawns),board=self))
            
        
        self.pieces_on_board.append(Rook(color, position=(0, row_others),board=self))
        self.pieces_on_board.append(Rook(color, position=(7, row_others),board=self))
        self.pieces_on_board.append(Knight(color, position=(1, row_others),board=self))
        self.pieces_on_board.append(Knight(color, position=(6, row_others),board=self))
        self.pieces_on_board.append(Bishop(color, position=(2, row_others),board=self))
        self.pieces_on_board.append(Bishop(color, position=(5, row_others),board=self))
        self.pieces_on_board.append(Queen(color, position=(3, row_others),board=self))
        self.pieces_on_board.append(King(color, position=(4, row_others),board=self))
            

    def reset_grid(self):
        self.grid = [[None for _ in range(8)] for _ in range(8)]

    
    def update_grid(self):
        self.reset_grid()
        for piece in self.pieces_on_board:
            x, y = piece._position
            self.grid[y][x] = piece


    def initialize(self):
        self.setup_pieces(Color.WHITE)
        self.setup_pieces(Color.BLACK)
        self.update_grid()
    
        
    def is_empty(self, square_coords):
        x, y = square_coords
        square = self.grid[y][x]
        return square == None
    

    def in_bounds(self, square_coords):
        x, y = square_coords
        return (0 <= x <= 7 and 0 <= y <= 7)
    

    def clone(self):
        return copy.deepcopy(self)
        
    
    def is_under_attack(self, color):
        return False



class Move():
    def __init__(self):
        self.captured_pieces = []
        self.move_history = []


        

class Piece(ABC):
    def __init__(self, color, position, board):
        self._color = color
        self._position = position
        self._has_moved = False
        self.board = board
    

    @abstractmethod
    def get_pseudo_legal_moves(self):
        pass
    
    
    def move(self, end_position):
        if self._move_is_legal(end_position):
            self._position = end_position
            self._has_moved = True


    def get_legal_moves(self): # noch nicht ganz fertig
        legal_moves = []
        for move in self.get_pseudo_legal_moves():
            x_old, y_old = self._position
            simulated_board = self.board.clone()
            simulated_board.grid[y_old][x_old]._position = move
            simulated_board.update_grid()

            if not simulated_board.is_under_attack(self._color):
                legal_moves.append(move)
        return legal_moves
    

    def _move_is_legal(self, end_position):
        return end_position in self.get_legal_moves()




class Pawn(Piece):
    def __init__(self, color, position, board):
        super().__init__(color, position, board)
        self._en_passant_vulnerability = False


    def get_pseudo_legal_moves(self):
        possible_moves = []

        direction = 1 if self._color == Color.WHITE else -1
        start_row = 1 if self._color == Color.WHITE else 6
        opp_color = Color.BLACK if self._color == Color.WHITE else Color.WHITE
        x, y = self._position

        # One step ahead
        if self.board.in_bounds((x, y+direction)):
            if self.board.is_empty((x, y+direction)):
                possible_moves.append((x, y+direction))
        
        # Two steps ahead
        if y == start_row and self.board.is_empty((x, y+2*direction)):
            possible_moves.append((x, y+2*direction))
        
        # Take piece
        if self.board.in_bounds((x+1, y+direction)):
            if not self.board.is_empty((x+1, y+direction)) and self.board.grid[y+direction][x+1]._color == opp_color:
                possible_moves.append((x+1, y+direction))
        if self.board.in_bounds((x-1, y+direction)):
            if not self.board.is_empty((x-1, y+direction)) and self.board.grid[y+direction][x-1]._color == opp_color:
                possible_moves.append((x-1, y+direction))
        
        # En passant
        if self.board.in_bounds((x+1, y+direction)):
            if self.board.grid[y][x+1] == Pawn and self.board.grid[y][x+1]._color == opp_color and self.board.is_empty((x+1, y+direction)):
                if self.board.grid[y][x+1]._en_passant_vulnerability:
                    possible_moves.append((x+1, y+direction))
        if self.board.in_bounds((x-1, y+direction)):
            if self.board.grid[y][x-1] == Pawn  and self.board.grid[y][x-1]._color == opp_color and self.board.is_empty((x+1, y+direction)):
                if self.board.grid[y][x-1]._en_passant_vulnerability:
                    possible_moves.append((x-1, y+direction))

        return possible_moves
    
    
    def move(self, end_position):
        if self._move_is_legal(end_position):
            _, y_old = self._position
            _, y_new = end_position
            direction = 1 if self._color == Color.WHITE else -1
            self._en_passant_vulnerability = True if (y_new-y_old) == 2*direction else False
            self._position = end_position
            self._has_moved = True
        



class Rook(Piece):
    def get_pseudo_legal_moves(self):
        possibel_moves = []
        directions = [(1, 0), (-1, 0), (0, 1), (0, -1)]
        opp_color = Color.BLACK if self._color == Color.WHITE else Color.WHITE

        for direction in directions:
            x_dir, y_dir = direction
            x_pos, y_pos = self._position
            while True:
                x_pos += x_dir
                y_pos += y_dir
                new_pos = (x_pos, y_pos)
                if not self.board.in_bounds(new_pos): break
                if not self.board.is_empty(new_pos): 
                    if self.board.grid[y_pos][x_pos]._color == opp_color: possibel_moves.append(new_pos)
                    break
                possibel_moves.append(new_pos)
        
        return possibel_moves




class Knight(Piece):
    def get_pseudo_legal_moves(self, board):
        possible_moves = []



class Bishop(Piece):
    def get_pseudo_legal_moves(self, board):
        pass



class Queen(Piece):
    def get_pseudo_legal_moves(self, board):
        pass



class King(Piece):
    def get_pseudo_legal_moves(self, board):
        pass


In [58]:
board = Board()
board.initialize()
board2 = board.clone()
board.grid[1][0]._position = (1, 1)
print(board.grid[1][0]._position)
print(board.pieces_on_board[0]._position)
print(board2.grid[1][0]._position)

(1, 1)
(1, 1)
(0, 1)


In [59]:
board, board2

(<__main__.Board at 0x10794c830>, <__main__.Board at 0x107f80910>)

In [60]:
board = Board()
board.initialize()
board.grid[1][0].get_legal_moves()

[(0, 2), (0, 3)]

In [61]:
board.grid[1][0].move((0, 2))
board.update_grid()
board.grid[2][0].move((0, 3))
board.update_grid()
print(board.grid)

[[<__main__.Rook object at 0x107f39a50>, <__main__.Knight object at 0x107f3a150>, <__main__.Bishop object at 0x107f3a250>, <__main__.Queen object at 0x107917820>, <__main__.King object at 0x107917950>, <__main__.Bishop object at 0x107f3a350>, <__main__.Knight object at 0x107f39450>, <__main__.Rook object at 0x107f39b50>], [None, <__main__.Pawn object at 0x107f795c0>, <__main__.Pawn object at 0x107f79630>, <__main__.Pawn object at 0x107f796a0>, <__main__.Pawn object at 0x107f79710>, <__main__.Pawn object at 0x107f79780>, <__main__.Pawn object at 0x107f797f0>, <__main__.Pawn object at 0x107f79860>], [None, None, None, None, None, None, None, None], [<__main__.Pawn object at 0x107f79550>, None, None, None, None, None, None, None], [None, None, None, None, None, None, None, None], [None, None, None, None, None, None, None, None], [<__main__.Pawn object at 0x107f798d0>, <__main__.Pawn object at 0x107f79940>, <__main__.Pawn object at 0x107f799b0>, <__main__.Pawn object at 0x107f79a20>, <__ma

In [62]:
board.grid[0][0].get_legal_moves()

[(0, 1), (0, 2)]